<a href="https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My lane is still Lane 2 (Refresh / Content Opportunity Scoring). The five plain-words contract answers:

1. **What one row means.** In the warehouse table, one row = one page on one day for one client (`report_date` × `client_hash_id` × `content_hash_id`). For my lane I add those daily rows up, so one *modelling* row = **one page** over a short window.
2. **Table I use.** `fact_content_daily_performance`, only the `month=2026-03` partition (a mid-panel month). I do **not** use the `_sample` table, because it is the final month (June 2026) and I am keeping that month sealed as a test month.
3. **Time window.** The decision moment is the end of 14 March 2026. Features come from 1–14 March. The outcome I check comes from 15–28 March. Two equal 14-day windows; 29–31 March are left unused.
4. **What I rank, and the label.** I rank pages by the risk that their impressions in the next 14 days fall more than 20% below the previous 14 days. This is an *observed later-window outcome*, not a rule applied to the current window. Only pages with at least 100 impressions in the feature window count.
5. **One thing I deliberately exclude.** Everything measured after the decision moment (the 15–28 March data) is excluded from the features. That is the label window, and using it as a feature would be leakage. Section 3 shows what that does.

In [9]:
%pip -q install duckdb
import os, getpass
import numpy as np
import pandas as pd
import duckdb

# Token order: env var -> Colab Secret named HF_TOKEN -> prompt (last resort).
# Never paste the token into a cell: this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
# One mid-panel month partition (NOT the _sample table, which is the final month).
M = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Contract windows (decision moment = end of 14 March 2026)
FEAT_START, FEAT_END   = "2026-03-01", "2026-03-14"
LABEL_START, LABEL_END = "2026-03-15", "2026-03-28"
MIN_IMPRESSIONS = 100

# Availability flag: use the search (GSC) flag if the table has one, else the GA4 flag.
cols = con.sql(f"SELECT * FROM {M} LIMIT 0").columns        # schema only, reads no rows
AVAIL = "gsc_data_available" if "gsc_data_available" in cols else "ga4_data_available"
print("Columns in the month partition:", cols)
print("Availability flag used:", AVAIL)

Columns in the month partition: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
Availability flag used: gsc_data_available


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| **Feature** (all from 1–14 March) | `imp_14d`, `ctr_pct_14d`, `avg_position_14d`, `active_days_14d`, `momentum_7d` | Knowable at the decision moment (each one is explained in section 3) |
| **Label** | `imp_label_14d` (impressions 15–28 March) → `declined` | The outcome I rank by. Never a feature |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date` | Grouping, joining and splitting only |
| **Excluded** | anything after 14 March as a feature; the `_sample` table; GA4 engagement columns | Future information; sealed test month; this slice runs on search data, and GA4 columns are zero-filled before a client's GA4 start |

**Missing values.** `avg_position_14d` is blank when a page has no position data, and `momentum_7d` is blank when week 1 had 0 impressions. I keep those blanks as blanks (no `fillna(0)`, which would make a gap look like a real value) and use a model that accepts them.

In [10]:
contract = {
    "feature":  ["imp_14d", "ctr_pct_14d", "avg_position_14d", "active_days_14d", "momentum_7d"],
    "label":    ["imp_label_14d", "declined"],
    "context":  ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["anything after 2026-03-14 as a feature", "_sample table", "GA4 engagement columns"],
}
all_cols = [c for v in contract.values() for c in v]
assert len(all_cols) == len(set(all_cols)), "a field is in two buckets"
pd.DataFrame({"bucket": list(contract), "fields": [", ".join(v) for v in contract.values()]})

,bucket,fields
0,feature,"imp_14d, ctr_pct_14d, avg_position_14d, active..."
1,label,"imp_label_14d, declined"
2,context,"client_hash_id, content_hash_id, report_date"
3,excluded,"anything after 2026-03-14 as a feature, _sampl..."


## 3. Verify it with queries (grain, counts, missing values, windows)

Three small queries on the March 2026 partition (grain, row count and date span, availability), then five features, then the leakage trap.

**Query 1 — grain.** Claim: one row = one page on one day for one client. If that holds, no (date, client, page) combination appears twice.

In [11]:
q1 = con.sql(f"""
    SELECT COUNT(*) AS duplicate_groups
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {M}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
print("Grain check: (report_date, client, page) groups with more than 1 row")
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check: (report_date, client, page) groups with more than 1 row


,duplicate_groups
0,0


**Query 2 — slice size and date span.** Claim: the slice is March 2026, so the dates should run 2026-03-01 to 2026-03-31.

In [12]:
q2 = con.sql(f"""
    SELECT COUNT(*)                        AS rows,
           MIN(report_date)                AS first_date,
           MAX(report_date)                AS last_date,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS pages
    FROM {M}
""").df()
q2

,rows,first_date,last_date,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 — availability.** Rows can have the flag TRUE, FALSE or NULL. I filter with `IS TRUE`, because `= TRUE` or `NOT FALSE` would mishandle the NULL rows. The query shows how many rows survive.

In [13]:
q3 = con.sql(f"""
    SELECT COUNT(*)                                       AS total_rows,
           COUNT(*) FILTER (WHERE {AVAIL} IS TRUE)        AS rows_available_true,
           COUNT(*) FILTER (WHERE {AVAIL} IS FALSE)       AS rows_false,
           COUNT(*) FILTER (WHERE {AVAIL} IS NULL)        AS rows_null
    FROM {M}
""").df()
q3["share_surviving_IS_TRUE"] = (q3["rows_available_true"] / q3["total_rows"]).round(3)
print(f"Availability flag: {AVAIL}")
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability flag: gsc_data_available


,total_rows,rows_available_true,rows_false,rows_null,share_surviving_IS_TRUE
0,9841378,3611061,6230317,0,0.367


**Five features** (all from 1–14 March, per page). Each one is knowable at the decision moment because it only uses days up to 14 March:

| Feature | What it is | Knowable at the decision moment because... |
|---|---|---|
| `imp_14d` | total search impressions, 1–14 Mar | it only adds up days that are already over by 14 March |
| `ctr_pct_14d` | clicks ÷ impressions × 100 (a percentage) | clicks and impressions are both from 1–14 March |
| `avg_position_14d` | average search position, 1–14 Mar (position 0 / blank = no data, left out) | it averages only positions recorded up to 14 March |
| `active_days_14d` | number of days with at least 1 impression, 1–14 Mar | it counts days that are already over |
| `momentum_7d` | impressions 8–14 Mar ÷ impressions 1–7 Mar | it compares two weeks that both end by 14 March |

The label is built from a separate window (15–28 March), in the same query but in its own column.

In [14]:
def window(a, b):
    return f"report_date BETWEEN DATE '{a}' AND DATE '{b}'"

frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           CAST(SUM(gsc_impressions) FILTER (WHERE {window(FEAT_START, FEAT_END)}) AS DOUBLE)  AS imp_14d,
           CAST(SUM(gsc_clicks)      FILTER (WHERE {window(FEAT_START, FEAT_END)}) AS DOUBLE)  AS clicks_14d,
           AVG(gsc_avg_position)     FILTER (WHERE {window(FEAT_START, FEAT_END)} AND gsc_avg_position > 0) AS avg_position_14d,
           COUNT(DISTINCT report_date) FILTER (WHERE {window(FEAT_START, FEAT_END)} AND gsc_impressions > 0) AS active_days_14d,
           CAST(SUM(gsc_impressions) FILTER (WHERE {window("2026-03-01", "2026-03-07")}) AS DOUBLE) AS imp_wk1,
           CAST(SUM(gsc_impressions) FILTER (WHERE {window("2026-03-08", "2026-03-14")}) AS DOUBLE) AS imp_wk2,
           CAST(SUM(gsc_impressions) FILTER (WHERE {window(LABEL_START, LABEL_END)}) AS DOUBLE) AS imp_label_14d
    FROM {M}
    WHERE {AVAIL} IS TRUE
    GROUP BY 1, 2
    HAVING imp_14d >= {MIN_IMPRESSIONS}
""").df()

frame["ctr_pct_14d"]   = 100 * frame["clicks_14d"] / frame["imp_14d"]
frame["momentum_7d"]   = frame["imp_wk2"] / frame["imp_wk1"].replace(0, np.nan)

# Label (observed later-window outcome). Assumption: no impressions recorded in the
# label window counts as 0 impressions.
n_no_label_rows = int(frame["imp_label_14d"].isna().sum())
frame["imp_label_14d"] = frame["imp_label_14d"].fillna(0)
frame["declined"] = (frame["imp_label_14d"] < 0.8 * frame["imp_14d"]).astype(int)

FEATURES = ["imp_14d", "ctr_pct_14d", "avg_position_14d", "active_days_14d", "momentum_7d"]
frame = frame[["client_hash_id", "content_hash_id"] + FEATURES + ["imp_label_14d", "declined"]]

print(f"{len(frame):,} pages (>= {MIN_IMPRESSIONS} impressions in 1-14 Mar); {n_no_label_rows:,} had no label-window rows (treated as 0).")
print(f"Share that declined >20% in 15-28 Mar: {frame['declined'].mean():.1%}")
frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

75,526 pages (>= 100 impressions in 1-14 Mar); 144 had no label-window rows (treated as 0).
Share that declined >20% in 15-28 Mar: 33.8%


,client_hash_id,content_hash_id,imp_14d,ctr_pct_14d,avg_position_14d,active_days_14d,momentum_7d,imp_label_14d,declined
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,211.0,0.473934,4.245075,14,0.610687,435.0,0
1,client_62f4a7e64f5e0096,content_d49a012dcb924e31,237.0,0.000000,4.542255,14,0.768657,77.0,1
2,client_62f4a7e64f5e0096,content_614baf2af4330bd7,385.0,0.259740,4.351876,14,0.896552,325.0,0
3,client_62f4a7e64f5e0096,content_225dc9235023be5f,272.0,0.367647,10.398054,14,1.000000,200.0,1
4,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,2484.0,0.000000,7.269611,14,0.863466,1393.0,1


In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(frame, feature_cols, k=50):
    """Hold out whole clients; return ROC AUC, precision@k and the base rate on the held-out clients."""
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
    tr, te = next(gss.split(frame, groups=frame["client_hash_id"]))
    y_tr, y_te = frame.iloc[tr]["declined"], frame.iloc[te]["declined"]
    model = HistGradientBoostingClassifier(max_iter=100, random_state=0)   # accepts blanks (NaN)
    model.fit(frame.iloc[tr][feature_cols], y_tr)
    p = pd.Series(model.predict_proba(frame.iloc[te][feature_cols])[:, 1], index=y_te.index)
    top = p.sort_values(ascending=False).head(k).index
    return roc_auc_score(y_te, p), y_te.loc[top].mean(), y_te.mean()

auc, p50, base = quick_score(frame, FEATURES)
print("HONEST score (5 features, held-out clients)")
print(f"  ROC AUC: {auc:.3f}   precision@50: {p50:.3f}   base rate: {base:.3f}")
honest = (auc, p50)

HONEST score (5 features, held-out clients)
  ROC AUC: 0.678   precision@50: 0.840   base rate: 0.322


**The trap.** I now add ONE column on purpose that is derived from the label: the label-window impressions divided by the feature-window impressions. That is exactly the number the label is made from. I expect the score to jump toward perfect. A score that good on a real prediction problem is a warning sign, not a win. Then I delete the column and keep the honest number above.

In [16]:
# 1) Add the label-derived column on purpose
frame["leak_label_ratio"] = frame["imp_label_14d"] / frame["imp_14d"]
auc_l, p50_l, base_l = quick_score(frame, FEATURES + ["leak_label_ratio"])
print("WITH the leak (looks amazing, means nothing)")
print(f"  ROC AUC: {auc_l:.3f}   precision@50: {p50_l:.3f}")

# 2) Delete it and keep the honest number
frame = frame.drop(columns=["leak_label_ratio"])
assert "leak_label_ratio" not in frame.columns
auc_h, p50_h, _ = quick_score(frame, FEATURES)
print("AFTER removing the leak (the honest number I keep)")
print(f"  ROC AUC: {auc_h:.3f}   precision@50: {p50_h:.3f}   (base rate {base_l:.3f})")
print(f"The leak added {auc_l - auc_h:+.3f} AUC. It only 'worked' because it contains the answer.")

WITH the leak (looks amazing, means nothing)
  ROC AUC: 1.000   precision@50: 1.000
AFTER removing the leak (the honest number I keep)
  ROC AUC: 0.678   precision@50: 0.840   (base rate 0.322)
The leak added +0.322 AUC. It only 'worked' because it contains the answer.


## 4. Data limits


**Named limitation of my slice: it is one month (March 2026), and the label is a 14-day vs 14-day change inside that month.** So my "decline" can't be told apart from seasonality or ordinary week-to-week noise, and anything I find may not hold in other months. It is also an unbalanced panel: clients have different history lengths, and some clients may supply most of the pages. A model that looks good here would still have to be tested on a later, sealed month (the final month), and a result here would be observational and decision-support only, not proof that a refresh helps.

In [17]:
# One number that backs the limitation: how concentrated is the slice in a few clients?
share = frame["client_hash_id"].value_counts(normalize=True)
print(f"Clients in the modelling frame: {share.size}")
print(f"Biggest client supplies {share.iloc[0]:.1%} of pages; top 3 supply {share.head(3).sum():.1%}.")
print("Slice = one month (2026-03); label = 14-day vs 14-day change inside it.")

Clients in the modelling frame: 37
Biggest client supplies 22.9% of pages; top 3 supply 55.4%.
Slice = one month (2026-03); label = 14-day vs 14-day change inside it.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.